In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {
    "User-Agent": "ProyectoUniversitarioUNALM/1.0 (contacto: taller2@unalm.edu.pe)"
}

FEEDS = {
    "Últimas Noticias": "https://elcomercio.pe/arc/outboundfeeds/rss/category/ultimas-noticias/?outputType=xml",
    "Política": "https://elcomercio.pe/arc/outboundfeeds/rss/category/politica/?outputType=xml",
    "Mundo": "https://elcomercio.pe/arc/outboundfeeds/rss/category/mundo/?outputType=xml",
}


def obtener_noticias_rss(nombre_categoria, url_feed):
    respuesta = requests.get(url_feed, headers=HEADERS, timeout=10)
    respuesta.raise_for_status()

    soup = BeautifulSoup(respuesta.content, "html.parser")
    items = soup.find_all("item")
    noticias = []

    for item in items:
        titulo = item.find("title").get_text().strip() if item.find("title") else ""
        descripcion = item.find("description").get_text().strip() if item.find("description") else ""
        enlace = item.find("link").get_text().strip() if item.find("link") else ""
        fecha = item.find("pubDate").get_text().strip() if item.find("pubDate") else ""

        noticias.append({
            "Categoria": nombre_categoria,
            "Título": titulo,
            "Descripción": descripcion,
            "Enlace": enlace,
            "Fecha_Publicacion": fecha
        })

    return noticias


print("Descargando feeds RSS de El Comercio...\n")

todas_las_noticias = []

for nombre_categoria, url_feed in FEEDS.items():
    print(f"  Procesando categoría: {nombre_categoria}")
    try:
        noticias = obtener_noticias_rss(nombre_categoria, url_feed)
        todas_las_noticias.extend(noticias)
        print(f"    -> {len(noticias)} noticias obtenidas")
    except requests.exceptions.RequestException as e:
        print(f"    Error al obtener el feed: {e}")
    time.sleep(1)

print(f"\nTotal de noticias recolectadas: {len(todas_las_noticias)}\n")

df = pd.DataFrame(todas_las_noticias)

if df.empty:
    print("No se pudo obtener información de ningún feed. Verifica tu conexión.")
else:
    PALABRAS_SENSACIONALISTAS = [
        "impactante", "increíble", "no lo vas a creer", "escándalo",
        "urgente", "alerta", "shock", "viral", "polémica", "explosivo"
    ]

    def analizar_titular(titulo):
        texto = titulo.lower()
        razones = []

        coincidencias = [p for p in PALABRAS_SENSACIONALISTAS if p in texto]
        if coincidencias:
            razones.append(f"Lenguaje sensacionalista ({len(coincidencias)} coincidencias)")

        letras = [c for c in titulo if c.isalpha()]
        if letras:
            ratio_mayusculas = sum(1 for c in letras if c.isupper()) / len(letras)
            if ratio_mayusculas > 0.5:
                razones.append("Exceso de mayúsculas")

        if titulo.count("!") >= 2 or titulo.count("?") >= 2:
            razones.append("Exceso de signos de exclamación/interrogación")

        return "; ".join(razones) if razones else "Ninguna"

    df["Señales_de_Sensacionalismo"] = df["Título"].apply(analizar_titular)
    df["¿Tiene_Señales?"] = df["Señales_de_Sensacionalismo"] != "Ninguna"

    print("=" * 70)
    print("   REPORTE: TITULARES DE EL COMERCIO POR CATEGORÍA")
    print("=" * 70)

    resumen_por_categoria = df.groupby("Categoria").agg(
        Total_Noticias=("Título", "count"),
        Con_Señales_Sensacionalistas=("¿Tiene_Señales?", "sum")
    ).reset_index()

    print("\nResumen por categoría:\n")
    print(resumen_por_categoria.to_string(index=False))

    tabla_detalle = df[["Categoria", "Título", "Fecha_Publicacion", "¿Tiene_Señales?", "Señales_de_Sensacionalismo"]]

    print("\n\nDetalle de todas las noticias analizadas:\n")
    print(tabla_detalle.to_string(index=False))

    df.to_csv("reporte_titulares_elcomercio.csv", index=False, encoding="utf-8")
    resumen_por_categoria.to_csv("resumen_categorias_elcomercio.csv", index=False, encoding="utf-8")

    print("\nDatos exportados a 'reporte_titulares_elcomercio.csv'")
    print("Resumen exportado a 'resumen_categorias_elcomercio.csv'")

Descargando feeds RSS de El Comercio...

  Procesando categoría: Últimas Noticias


C:\Users\User\AppData\Local\Temp\ipykernel_26504\2159603294.py:21: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(respuesta.content, "html.parser")


    -> 0 noticias obtenidas
  Procesando categoría: Política
    -> 66 noticias obtenidas
  Procesando categoría: Mundo
    -> 95 noticias obtenidas

Total de noticias recolectadas: 161

   REPORTE: TITULARES DE EL COMERCIO POR CATEGORÍA

Resumen por categoría:

Categoria  Total_Noticias  Con_Señales_Sensacionalistas
    Mundo              95                             1
 Política              66                             0


Detalle de todas las noticias analizadas:

Categoria                                                                                                                                                        Título Fecha_Publicacion  ¿Tiene_Señales?                 Señales_de_Sensacionalismo
 Política                    <![CDATA[Ausencias, gestos y silencios: el detrás de cámaras de la entrega de credenciales a los senadores del nuevo Congreso bicameral]]>                              False                                    Ninguna
 Política                       